# 🚀 DataMate: 1-Click Evaluation & Runner on Google Colab Pro

This notebook provides a seamless, 1-click execution of the **DataMate** agent pipeline using **vLLM** on high-performance Google Colab Pro GPUs (**NVIDIA A100 SXM 40GB/80GB** or **L4 24GB**).

### 💡 Key Benefits of Running on Colab Pro A100:
- **Single-GPU Peak Throughput (`tp=1`):** Eliminates inter-GPU PCIe tensor parallel latency, running at ~80–120 tokens/sec.
- **10x Faster Execution:** 200 questions benchmark completes in **~15–20 minutes** (compared to 3.5 hours on dual A30s).
- **Generous High-RAM & VRAM:** Ample space for `Qwen2.5-Coder-14B-Instruct-GPTQ-Int8` + 32k KV Cache.
- **Automatic Result Evaluation:** Instant calculation of Penalized Accuracy (PA) and automated breakdown of errors.

--- 
> **⚠️ Important Runtime Setup:**
> In the menu above, go to **Runtime → Change runtime type**:
> - **Hardware accelerator:** `A100 GPU` (or `L4 GPU`)
> - **Runtime shape:** `High-RAM`
> Click **Save**, then run the cells in order.

## 1. Hardware & GPU Acceleration Verification

In [ ]:
# Check NVIDIA GPU allocation
!nvidia-smi

import torch
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model    : {device_name}")
    print(f"Total VRAM   : {total_vram:.2f} GB")
    print(f"Compute Cap  : {torch.cuda.get_device_capability(0)}")
    if total_vram >= 22:
        print("✅ Excellent! Sufficient VRAM for Qwen2.5-Coder-14B with full KV cache.")
    else:
        print("⚠️ Warning: Less than 22 GB VRAM detected. L4 or A100 is recommended.")
print("=" * 60)

## 2. Mount Google Drive (Recommended for Persistence)
Mounting Google Drive allows caching the model weights and persisting submission files across Colab restarts.

In [ ]:
from google.colab import drive
import os

drive_path = "/content/drive/MyDrive/datamate"
try:
    drive.mount('/content/drive')
    os.makedirs(drive_path, exist_ok=True)
    print(f"✅ Google Drive connected! Persistence folder: {drive_path}")
except Exception as e:
    print(f"ℹ️ Drive mount skipped: {e}. Outputs will be stored in /content/result.")

## 3. Clone Repository & Setup Environment

In [ ]:
import os

REPO_URL = "https://github.com/vohoangloc-32/datamate.git"
BRANCH = "loc"
WORKDIR = "/content/datamate"

if not os.path.exists(WORKDIR):
    print(f"Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL} {WORKDIR}
else:
    print(f"Pulling latest updates in {WORKDIR}...")
    !cd {WORKDIR} && git fetch origin && git checkout {BRANCH} && git reset --hard origin/{BRANCH}

%cd {WORKDIR}
!git log -1 --oneline


## 4. Install Dependencies & Run Verification Tests

In [ ]:
# Install vLLM and DataMate requirements
!pip install -q vllm openpyxl xlrd tabulate pytest huggingface_hub pandas

# Run full unit test suite to verify pipeline integrity
!pytest tests/ -q

## 5. Download Model Weights to Local NVMe
Downloads `Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8` (high-speed parallel download via `huggingface_hub`).

In [ ]:
from huggingface_hub import snapshot_download
import os
import shutil

MODEL_REPO = "Qwen/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
LOCAL_MODEL_DIR = "/content/model_cache/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"
DRIVE_MODEL_BACKUP = "/content/drive/MyDrive/datamate/models/Qwen2.5-Coder-14B-Instruct-GPTQ-Int8"

if os.path.exists(DRIVE_MODEL_BACKUP) and len(os.listdir(DRIVE_MODEL_BACKUP)) > 5:
    print(f"Found model in Google Drive backup. Copying to local NVMe SSD...")
    os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
    !cp -r {DRIVE_MODEL_BACKUP}/* {LOCAL_MODEL_DIR}/
else:
    print(f"Downloading {MODEL_REPO} from Hugging Face Hub (fast parallel download)... ")
    snapshot_download(
        repo_id=MODEL_REPO,
        local_dir=LOCAL_MODEL_DIR,
        local_dir_use_symlinks=False,
        resume_download=True,
        max_workers=8,
    )
    # Optional backup to Google Drive for instantaneous setup on future runs
    if os.path.exists("/content/drive/MyDrive/datamate"):
        print("Backing up model to Google Drive for future instant loads...")
        os.makedirs(DRIVE_MODEL_BACKUP, exist_ok=True)
        !cp -rn {LOCAL_MODEL_DIR}/* {DRIVE_MODEL_BACKUP}/

print(f"✅ Model ready at: {LOCAL_MODEL_DIR}")
!ls -lh {LOCAL_MODEL_DIR}

## 6. Run DataMate Prediction Pipeline
Executes the full pipeline with calibrated SOTA ConfidenceGate and AST Granularity Refiner.
- **Dataset:** `data/train`
- **Questions:** `data/sample_questions_500.jsonl` (500 benchmark questions) or `data/sample_questions_200.jsonl` (200 questions)
- **Confidence Threshold:** `0.45`


In [ ]:
import os
import time

# Environment optimizations for single A100 / L4 GPU
os.environ["VLLM_TP"] = "1"           # Single GPU = 0 inter-GPU overhead
os.environ["VLLM_DTYPE"] = "auto"     # Native GPTQ activation dtype
os.environ["VLLM_MAX_MODEL_LEN"] = "16384"
os.environ["VLLM_MAX_NEW_TOKENS"] = "1024"

# Dataset configuration
DATA_DIR = "/content/datamate/data/train"
# Choose question set:
QUESTIONS_FILE = "/content/datamate/data/sample_questions_500.jsonl"  # 500 questions benchmark (~20-25 mins)
# QUESTIONS_FILE = "/content/datamate/data/sample_questions_200.jsonl"  # 200 questions benchmark (~8-12 mins)
# QUESTIONS_FILE = "/content/datamate/data/public_test/questions.jsonl" # Public test set (1000 questions)
OUTPUT_DIR = "/content/result"
CONFIDENCE_THRESHOLD = "0.45"

print("=" * 60)
print(f"Starting DataMate Pipeline...")
print(f"Data Directory : {DATA_DIR}")
print(f"Questions File : {QUESTIONS_FILE}")
print(f"Output Dir     : {OUTPUT_DIR}")
print(f"Model Dir      : {LOCAL_MODEL_DIR}")
print("=" * 60)

t_start = time.time()
!python src/predict.py \
    --data-dir {DATA_DIR} \
    --questions-file {QUESTIONS_FILE} \
    --output-dir {OUTPUT_DIR} \
    --model-dir {LOCAL_MODEL_DIR} \
    --confidence-threshold {CONFIDENCE_THRESHOLD} \
    --log-level INFO

elapsed = time.time() - t_start
print(f"Pipeline finished in {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)!")


## 7. Compute Official Penalized Accuracy (PA) & Benchmark Results
Evaluates predictions against ground truth answers using the competition metric:
$$\text{PA} = \frac{\text{Correct} - \text{Wrong}}{\text{Total}}$$

In [ ]:
import os
import json
from src.evaluate import compute_pa

SUBMISSION_PATH = os.path.join(OUTPUT_DIR, "submission.jsonl")
ANSWERS_PATH = os.path.join(DATA_DIR, "answers.jsonl")

if os.path.exists(SUBMISSION_PATH) and os.path.exists(ANSWERS_PATH):
    preds = {json.loads(l)["id"]: json.loads(l) for l in open(SUBMISSION_PATH, encoding="utf-8-sig") if l.strip()}
    refs = {json.loads(l)["id"]: json.loads(l) for l in open(ANSWERS_PATH, encoding="utf-8-sig") if l.strip() and json.loads(l)["id"] in preds}
    metrics = compute_pa(preds, refs)
    total = metrics["total"]
    correct = metrics["correct"]
    wrong = metrics["wrong"]
    abstain = metrics["abstain"]
    pa = metrics["pa"]
    
    print("\n" + "=" * 60)
    print("🏆 DATAMATE OFFICIAL BENCHMARK SCORE (EVALUATED SUBSET)")
    print("=" * 60)
    print(f"Total Questions  : {total}")
    print(f"Correct (+1 pt)  : {correct} ({correct/total*100:.1f}%)")
    print(f"Wrong   (-1 pt)  : {wrong} ({wrong/total*100:.1f}%)")
    print(f"Abstained (0 pt) : {abstain} ({abstain/total*100:.1f}%)")
    print("-" * 60)
    print(f"🔥 Penalized Accuracy (PA): {pa:+.4f} ({pa*100:+.2f}%)")
    print("=" * 60)
    
    # Save backup to Google Drive if available
    if os.path.exists("/content/drive/MyDrive/datamate"):
        !cp -f {SUBMISSION_PATH} /content/drive/MyDrive/datamate/submission_colab.jsonl
        print("💾 Saved submission copy to /content/drive/MyDrive/datamate/submission_colab.jsonl")
else:
    print(f"ℹ️ Submission or answers file not found. Check {SUBMISSION_PATH}")


## 8. Diagnostic Inspector: Error Analysis Breakdown

In [ ]:
import json
import pandas as pd

# Load predictions and ground truths
preds = {}
with open(SUBMISSION_PATH, "r", encoding="utf-8-sig") as f:
    for line in f:
        row = json.loads(line)
        preds[row["id"]] = row

truths = {}
with open(ANSWERS_PATH, "r", encoding="utf-8-sig") as f:
    for line in f:
        row = json.loads(line)
        truths[row["id"]] = row

wrong_list = []
abstain_list = []

for qid, p in preds.items():
    if qid not in truths:
        continue
    g = truths[qid]
    p_val, g_val = p["answer"], g["answer"]
    p_typ, g_typ = p.get("answer_type"), g.get("answer_type")
    
    if p_typ == "abstain":
        abstain_list.append({"id": qid, "gt_type": g_typ, "gt_val": str(g_val)[:60]})
    elif p_typ != g_typ or str(p_val) != str(g_val):
        wrong_list.append({
            "id": qid,
            "pred_type": p_typ,
            "gt_type": g_typ,
            "pred_val": str(p_val)[:60],
            "gt_val": str(g_val)[:60]
        })

print(f"Total Errors: {len(wrong_list)} | Total Abstains: {len(abstain_list)}")
if wrong_list:
    df_err = pd.DataFrame(wrong_list)
    print("\n--- First 15 Incorrect Questions ---")
    display(df_err.head(15))
    
    # Save error analysis report
    df_err.to_csv("/content/result/error_analysis.csv", index=False, encoding="utf-8-sig")
    print("📄 Saved error report to /content/result/error_analysis.csv")


## 9. Backup All Results & Logs to Google Drive
Saves all prediction outputs, submission files, error analysis reports (`error_analysis.csv`), and execution logs (`predict.log`) to a timestamped folder and zip archive in Google Drive (`/content/drive/MyDrive/datamate/results/`).


In [ ]:
import os
import shutil
from datetime import datetime

DRIVE_BASE_DIR = "/content/drive/MyDrive/datamate"
RESULT_DIR = "/content/result"

if not os.path.exists("/content/drive/MyDrive"):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"⚠️ Google Drive mount failed: {e}")

if os.path.exists(RESULT_DIR):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_drive_root = os.path.join(DRIVE_BASE_DIR, "results")
    timestamped_dir = os.path.join(results_drive_root, f"run_{timestamp}")
    latest_dir = os.path.join(results_drive_root, "latest")
    os.makedirs(timestamped_dir, exist_ok=True)
    os.makedirs(latest_dir, exist_ok=True)
    
    # 1. Copy all result files to timestamped directory & latest directory
    for item in os.listdir(RESULT_DIR):
        s = os.path.join(RESULT_DIR, item)
        d_time = os.path.join(timestamped_dir, item)
        d_late = os.path.join(latest_dir, item)
        if os.path.isdir(s):
            shutil.copytree(s, d_time, dirs_exist_ok=True)
            shutil.copytree(s, d_late, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d_time)
            shutil.copy2(s, d_late)
            
    # 2. Create a compressed zip backup
    zip_base = os.path.join(results_drive_root, f"results_{timestamp}")
    shutil.make_archive(zip_base, "zip", RESULT_DIR)
    
    print("=" * 60)
    print("✅ Successfully backed up all results to Google Drive!")
    print(f"📁 Timestamped Folder : {timestamped_dir}")
    print(f"📁 Latest Folder      : {latest_dir}")
    print(f"📦 Zip Archive        : {zip_base}.zip")
    print("=" * 60)
    !ls -lh {timestamped_dir}
else:
    print(f"❌ Result directory not found: {RESULT_DIR}")


## 10. 1-Click Submission File Download
Download the final `submission.jsonl` to your local machine for competition submission.


In [ ]:
from google.colab import files
if os.path.exists(SUBMISSION_PATH):
    files.download(SUBMISSION_PATH)
    print(f"✅ Downloading {SUBMISSION_PATH}...")
else:
    print("❌ File not found.")